# Part 17: Pretrain (pIC50 regression on ChEMBL 4,441) → Finetune (Hybrid on curated 645)

**Strategy:** Pretrain Hybrid (FP + GCN) to predict **pIC50 regression** on ChEMBL 4,441 (CHEMBL5023, IC50/Ki/Kd/EC50, exact, nM), then finetune classification on curated 645 (RO5, high-quality). Fixes prior-collapse: broad chemical space from pretrain + correct decision boundary from curated.

**Strict anti-leakage:** Pretrain excludes *any* scaffold present in finetune test (Bemis-Murcko) + exact overlap (537 IDs). Two splits reported: random (comparability) + scaffold (generalization). Decisions on scaffold.

**No teacher:** Pure hybrid, RF benchmark only.

**Folders:** `Part_17/data, models, results` — VS Code ↔ Colab portable (path candidates, conditional clone).

**Data:** `Part_2/ro5_properties_filtered.csv` (645) + `Part_12/mdm2_chembl_clean.csv` (4,441) → COCONUT `Part_8/screening_results.csv` (~154k)
**Seeds:** 57
**Labels (finetune):** Active pIC50 ≥6, Inactive <5 (intermediates dropped for classification; regression uses continuous pIC50)



In [ ]:
import os, sys, pathlib, subprocess, importlib
# --- locate repo root & Part_2/Part_12 files (handles top-level new_method vs subfolder Natural_MDM2) ---
CANDID_RO5 = [
    "Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv",
    "Part_2/ro5_properties_filtered.csv",
    "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv",
    "Part_10/ro5_properties_filtered.csv",
]
CANDID_CHEMBL = [
    "Natural_MDM2_Inhibitor_Discovery_using_ML/Part_12/mdm2_chembl_clean.csv",
    "Part_12/mdm2_chembl_clean.csv",
    "../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_12/mdm2_chembl_clean.csv",
]
def find_first(cands):
    for p in cands:
        if os.path.exists(p):
            return p
    # glob fallback
    import glob
    for pat in ["**/ro5_properties_filtered.csv","**/mdm2_chembl_clean.csv"]:
        for f in glob.glob(pat, recursive=True):
            if os.path.exists(f):
                return f
    return None

p645 = find_first(CANDID_RO5)
p4441 = find_first(CANDID_CHEMBL)
print(f"p645 candidate: {p645} exists={os.path.exists(p645) if p645 else False}")
print(f"p4441 candidate: {p4441} exists={os.path.exists(p4441) if p4441 else False}")

# if missing, fetch from ChEMBL API (CHEMBL5023) like Part_16 did — writes to Part_17/data/mdm2_chembl_raw.csv then cleans
if not p4441 or not os.path.exists(p4441):
    print("ChEMBL clean not found — will fetch via API in next cell")
else:
    print("found")

# ensure Part_17 dirs regardless of cwd
for d in ["Part_17/data","Part_17/models","Part_17/results"]:
    os.makedirs(d, exist_ok=True)
print("Part_17", os.listdir("Part_17") if os.path.exists("Part_17") else "check ../Part_17", os.listdir("../Part_17") if os.path.exists("../Part_17") else "")
# normalize cwd to new_method root (where Part_17 lives)
if os.path.exists("Part_17/data"):
    ROOT="."
elif os.path.exists("../Part_17/data"):
    os.chdir("..")
    ROOT="."
else:
    ROOT="."
print(f"ROOT={os.path.abspath(ROOT)}")
# re-resolve after chdir
p645 = find_first(CANDID_RO5)
p4441 = find_first(CANDID_CHEMBL)
print(f"after chdir p645={p645} p4441={p4441}")



In [ ]:
# pip ensure (quiet) — uncomment if needed in Colab
# import subprocess, sys
# subprocess.check_call([sys.executable,"-m","pip","install","-q","torch-geometric","rdkit"])
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, random, warnings, os, pickle
warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Lipinski, AllChem, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, matthews_corrcoef, classification_report, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier
from tqdm import tqdm
from collections import defaultdict

SEED=57
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device} seed={SEED}")



## 1. Load datasets (645 curated + 4,441 ChEMBL) — fetch if missing


In [ ]:
import glob, time, requests

# prefer already-resolved p645/p4441 from previous cell; re-find if needed
if 'p645' not in globals() or not p645 or not os.path.exists(p645):
    cands=["Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv","Part_2/ro5_properties_filtered.csv","../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_2/ro5_properties_filtered.csv"]
    for c in cands:
        if os.path.exists(c):
            p645=c; break
if 'p4441' not in globals() or not p4441 or not os.path.exists(p4441):
    cands2=["Natural_MDM2_Inhibitor_Discovery_using_ML/Part_12/mdm2_chembl_clean.csv","Part_12/mdm2_chembl_clean.csv"]
    for c in cands2:
        if os.path.exists(c):
            p4441=c; break

print(f"using p645={p645}")
print(f"using p4441={p4441}")

# if still missing, fetch from ChEMBL API (13k raw → clean)
if not p4441 or not os.path.exists(p4441):
    print("Fetching CHEMBL5023 from API (≈13k activities)...")
    base="https://www.ebi.ac.uk/chembl/api/data/activity.json"
    target="CHEMBL5023"
    all_acts=[]; offset=0; limit=1000
    while True:
        url=f"{base}?target_chembl_id={target}&limit={limit}&offset={offset}"
        r=requests.get(url, timeout=60)
        r.raise_for_status()
        j=r.json()
        acts=j.get('activities',[])
        if not acts: break
        all_acts.extend(acts)
        total=j['page_meta']['total_count']
        print(f" fetched {len(all_acts)}/{total}")
        if j['page_meta']['next'] is None: break
        offset+=limit; time.sleep(0.3)
    df_raw=pd.DataFrame(all_acts)
    df_raw.to_csv("Part_17/data/mdm2_raw_chembl.csv", index=False)
    print(f"raw saved {len(df_raw)}")
    # clean like Part_16
    df=df_raw.copy()
    keep_types=['IC50','Ki','Kd','EC50']
    df=df[df['standard_type'].isin(keep_types)]
    df=df[df['standard_relation']=='=']
    df=df[df['standard_units']=='nM']
    df=df[df['standard_value'].notna()]
    import numpy as np
    df['standard_value']=pd.to_numeric(df['standard_value'], errors='coerce')
    df=df[df['standard_value'].notna() & (df['standard_value']>0)]
    df=df.sort_values('standard_value').drop_duplicates('molecule_chembl_id', keep='first')
    df['pIC50']=-np.log10(df['standard_value']*1e-9)
    df['bioactivity_class']='Ambiguous'
    df.loc[df['standard_value']<=1000,'bioactivity_class']='Active'
    df.loc[df['standard_value']>10000,'bioactivity_class']='Inactive'
    df=df[df['bioactivity_class'].isin(['Active','Inactive'])]
    # valid smiles
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Lipinski
    valid=[]
    for smi in df['canonical_smiles']:
        valid.append(Chem.MolFromSmiles(str(smi)) is not None)
    df=df[valid]
    # props
    mw, logp, hba, hbd=[],[],[],[]
    for smi in df['canonical_smiles']:
        m=Chem.MolFromSmiles(str(smi))
        mw.append(Descriptors.MolWt(m)); logp.append(Descriptors.MolLogP(m)); hba.append(Lipinski.NumHAcceptors(m)); hbd.append(Lipinski.NumHDonors(m))
    df['molecular_weight']=mw; df['logp']=logp; df['n_hba']=hba; df['n_hbd']=hbd; df['valid_smile']=True
    df['action_type']=None
    df.to_csv("Part_17/data/mdm2_chembl_clean.csv", index=False)
    p4441="Part_17/data/mdm2_chembl_clean.csv"
    print(f"clean saved {len(df)} {df['bioactivity_class'].value_counts().to_dict()}")
    print("using p4441",p4441)

df645=pd.read_csv(p645)
df4441=pd.read_csv(p4441, low_memory=False)
print(f"645: {len(df645)} {df645['bioactivity_class'].value_counts().to_dict()}")
print(f"4441: {len(df4441)} {df4441['bioactivity_class'].value_counts().to_dict() if 'bioactivity_class' in df4441.columns else df4441.shape}")
print("645 pIC50", df645['pIC50'].describe().to_dict())
print("4441 pIC50", df4441['pIC50'].describe().to_dict() if 'pIC50' in df4441.columns else 'no')
df645.head(3)



## 2. Curated 645 scaffold split (finetune test is sacred — never in pretrain)


In [ ]:
def scaffold_of(smiles):
    m=Chem.MolFromSmiles(str(smiles))
    if m is None: return "__invalid__"
    try:
        scaf=MurckoScaffold.GetScaffoldForMol(m)
        return Chem.MolToSmiles(scaf) if scaf.GetNumAtoms()>0 else "__no_scaffold__"
    except: return "__error__"

# curated scaffolds
cur_scaffolds=[scaffold_of(s) for s in df645['canonical_smiles']]
print(f"curated unique scaffolds: {len(set(cur_scaffolds))} / {len(cur_scaffolds)}")
# scaffold split: group by scaffold, sort by size, fill test 20%
from collections import defaultdict, Counter
sc_to_idx=defaultdict(list)
for i,sc in enumerate(cur_scaffolds):
    sc_to_idx[sc].append(i)
groups=sorted(sc_to_idx.values(), key=len, reverse=True)
target_te=int(0.2*len(df645))
te_scaf=set()
for g in groups:
    if len(te_scaf)+len(g) <= target_te+5:
        te_scaf.update(g)
    if len(te_scaf)>=target_te: break
if len(te_scaf)<target_te:
    remaining=[i for i in range(len(df645)) if i not in te_scaf]
    need=target_te-len(te_scaf)
    te_scaf.update(np.random.RandomState(SEED).choice(remaining, need, replace=False).tolist())
te_cur=np.array(sorted(te_scaf))
tr_cur=np.array([i for i in range(len(df645)) if i not in set(te_cur.tolist())])
print(f"curated scaffold split: train {len(tr_cur)} test {len(te_cur)}")
print(f" train scaffold {len(set(cur_scaffolds[i] for i in tr_cur))} test scaffold {len(set(cur_scaffolds[i] for i in te_cur))} overlap {len(set(cur_scaffolds[i] for i in tr_cur)&set(cur_scaffolds[i] for i in te_cur))}")
# also random split for comparability
idx=np.arange(len(df645))
tr_rand, te_rand = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=df645['bioactivity_class'].map({'Active':1,'Inactive':0,'active':1,'inactive':0}).fillna(0).astype(int) if 'bioactivity_class' in df645.columns else None)
print(f"random split: train {len(tr_rand)} test {len(te_rand)}")

# save
np.savez("Part_17/data/splits_curated.npz", tr_cur=tr_cur, te_cur=te_cur, tr_rand=tr_rand, te_rand=te_rand, cur_scaffolds=np.array(cur_scaffolds, dtype=object))
print("saved Part_17/data/splits_curated.npz")

# for pretrain exclusion: set of test scaffolds (scaffold split)
test_scaffolds=set(cur_scaffolds[i] for i in te_cur)
test_smiles=set(df645.iloc[te_cur]['canonical_smiles'].astype(str))
test_ids=set(df645.iloc[te_cur]['molecule_chembl_id'].astype(str)) if 'molecule_chembl_id' in df645.columns else set()
print(f"test scaffolds to exclude from pretrain: {len(test_scaffolds)}  test ids {len(test_ids)}")



## 3. Build pretrain set (ChEMBL 4,441 → exclude test scaffolds + overlap → pIC50 regression)


In [ ]:
# pretrain uses continuous pIC50, not just active/inactive — keep intermediates too if present (we dropped them in clean, so only act/inact remain; that's okay, regression still spans ~4 to 11)
# but we should use the full cleaned df4441 which already dropped ambiguous? For regression we ideally want intermediates too. However Part_12 clean dropped ambiguous (533). Let's keep as is for now (still wide range due to act/inact spanning -0.2 to 11.7). To recover intermediates we could re-fetch raw and keep all, but keep simple: use 4441 as pretrain (still regression on pIC50, not binary).

df_pre = df4441.copy()
print(f"pre raw {len(df_pre)}")
# exclude exact IDs overlapping with curated test
if 'molecule_chembl_id' in df_pre.columns:
    mask_id = df_pre['molecule_chembl_id'].astype(str).isin(test_ids)
    print(f"overlap IDs {mask_id.sum()} will be removed")
    df_pre = df_pre[~mask_id]
# exclude scaffold overlap
pre_scaffolds=[scaffold_of(s) for s in df_pre['canonical_smiles']]
mask_scaf=np.array([sc in test_scaffolds for sc in pre_scaffolds])
print(f"scaffold overlap {mask_scaf.sum()} will be removed")
df_pre=df_pre[~mask_scaf].reset_index(drop=True)
print(f"pretrain after exclusion: {len(df_pre)} (was {len(df4441)})")
print(f" pIC50 {df_pre['pIC50'].describe().to_dict()}")
print(df_pre['bioactivity_class'].value_counts().to_dict() if 'bioactivity_class' in df_pre.columns else 'no class')

# also create holdout for pretrain validation (10% scaffold)
# simple random 10% for regression validation (not critical)
pre_idx=np.arange(len(df_pre))
pre_tr, pre_val = train_test_split(pre_idx, test_size=0.1, random_state=SEED)
print(f"pretrain split: train {len(pre_tr)} val {len(pre_val)}")
np.savez("Part_17/data/splits_pretrain.npz", pre_tr=pre_tr, pre_val=pre_val, df_pre_len=len(df_pre))
df_pre.head(2)



## 4. Featurization (Morgan 2048 + MACCS 167 = 2215 when USE_MACCS=True + 78-dim graph) — same as Part 16


In [ ]:
# Fingerprints: Morgan uses the modern rdFingerprintGenerator API (RDKit >= 2023.03).
# No deprecated AllChem.GetMorganFingerprintAsBitVect calls. MACCSkeys.GenMACCSKeys
# remains the supported MACCS API (rdFingerprintGenerator has no MACCS generator).
# Toggle USE_MACCS to include (2215-dim) or exclude (2048-dim) MACCS keys.
USE_MACCS = True
MORGAN_NBITS = 2048
MACCS_DIM = 167
FP_DIM = (MORGAN_NBITS + MACCS_DIM) if USE_MACCS else MORGAN_NBITS

from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
_morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=MORGAN_NBITS)
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


def morgan_fp(smi, nBits=MORGAN_NBITS):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    bv = _morgan_gen.GetFingerprint(m)
    arr = np.zeros((nBits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr


def maccs_fp(smi):
    from rdkit.Chem import MACCSkeys
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    bv = MACCSkeys.GenMACCSKeys(m)
    arr = np.zeros((MACCS_DIM,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(bv, arr)
    return arr


def combined_fp(smi):
    a = morgan_fp(smi)
    if a is None:
        return None
    if USE_MACCS:
        b = maccs_fp(smi)
        if b is None:
            return None
        return np.concatenate([a, b]).astype(np.float32)
    return a

ATOM_CHOICES={'atomic_num':list(range(1,101)),'degree':[0,1,2,3,4,5],'formal_charge':[-2,-1,0,1,2,3],'num_hs':[0,1,2,3,4],'hybridization':[Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2, Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D, Chem.rdchem.HybridizationType.SP3D2]}
def one_hot(v,choices):
    enc=[0]*len(choices)
    if v in choices: enc[choices.index(v)]=1
    return enc
def atom_features(atom):
    f=[]
    f+=one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f+=one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f+=one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f+=one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f+=one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic())); f.append(int(atom.IsInRing()))
    return (f+[0]*78)[:78]
def mol_to_graph(smiles, label=None, pIC50=None):
    m=Chem.MolFromSmiles(str(smiles))
    if m is None: return None
    x=torch.tensor([atom_features(a) for a in m.GetAtoms()], dtype=torch.float)
    ei=[]
    for b in m.GetBonds():
        i,j=b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        ei.extend([[i,j],[j,i]])
    if not ei: ei=[[0,0]]
    ei=torch.tensor(ei, dtype=torch.long).t().contiguous()
    d=Data(x=x, edge_index=ei)
    if label is not None:
        d.y=torch.tensor([label], dtype=torch.long)
    if pIC50 is not None:
        d.y_reg=torch.tensor([pIC50], dtype=torch.float)
    return d

# build curated graphs/fps/labels
label_map={'Active':1,'Inactive':0}
cur_graphs=[]; cur_fps=[]; cur_labels=[]; cur_pIC50=[]; cur_valid_idx=[]
for i,row in tqdm(df645.iterrows(), total=len(df645), desc='curated featurize'):
    smi=row['canonical_smiles']
    fp=combined_fp(smi)
    lab=label_map.get(row['bioactivity_class'], None)
    pic=row['pIC50']
    g=mol_to_graph(smi, label=lab, pIC50=pic)
    if fp is not None and g is not None and lab is not None:
        cur_fps.append(fp); cur_graphs.append(g); cur_labels.append(lab); cur_pIC50.append(pic); cur_valid_idx.append(i)
cur_fps=np.stack(cur_fps); cur_labels=np.array(cur_labels); cur_pIC50=np.array(cur_pIC50, dtype=np.float32)
print(f"curated valid {len(cur_valid_idx)}/{len(df645)} fps {cur_fps.shape} graphs {len(cur_graphs)}")
# map original idx -> new position
orig_to_new={orig:new for new,orig in enumerate(cur_valid_idx)}
# translate splits from original df indices to new positions (drop invalid if any)
def translate(idxs):
    return np.array([orig_to_new[i] for i in idxs if i in orig_to_new], dtype=int)
tr_cur_new=translate(tr_cur); te_cur_new=translate(te_cur); tr_rand_new=translate(tr_rand); te_rand_new=translate(te_rand)
print(f"translated: tr_cur {len(tr_cur_new)}/{len(tr_cur)} te_cur {len(te_cur_new)}/{len(te_cur)}")
np.savez("Part_17/data/featurized_curated.npz", cur_fps=cur_fps, cur_labels=cur_labels, cur_pIC50=cur_pIC50, cur_valid_idx=np.array(cur_valid_idx), tr_cur_new=tr_cur_new, te_cur_new=te_cur_new, tr_rand_new=tr_rand_new, te_rand_new=te_rand_new)

# pretrain graphs/fps/pIC50
pre_graphs=[]; pre_fps=[]; pre_pIC50_list=[]; pre_valid_idx=[]
for i,row in tqdm(df_pre.iterrows(), total=len(df_pre), desc='pretrain featurize'):
    smi=row['canonical_smiles']
    fp=combined_fp(smi)
    pic=float(row['pIC50'])
    g=mol_to_graph(smi, pIC50=pic)
    if fp is not None and g is not None and not np.isnan(pic):
        pre_fps.append(fp); pre_graphs.append(g); pre_pIC50_list.append(pic); pre_valid_idx.append(i)
pre_fps=np.stack(pre_fps) if pre_fps else np.zeros((0,FP_DIM), dtype=np.float32)
pre_pIC50_arr=np.array(pre_pIC50_list, dtype=np.float32)
print(f"pretrain valid {len(pre_valid_idx)}/{len(df_pre)} fps {pre_fps.shape} graphs {len(pre_graphs)}")
# translate pre_tr/pre_val (indices into df_pre) to new positions
pre_orig_to_new={orig:new for new,orig in enumerate(pre_valid_idx)}
pre_tr_new=np.array([pre_orig_to_new[i] for i in pre_tr if i in pre_orig_to_new], dtype=int)
pre_val_new=np.array([pre_orig_to_new[i] for i in pre_val if i in pre_orig_to_new], dtype=int)
print(f"pretrain translated: tr {len(pre_tr_new)} val {len(pre_val_new)}")
np.savez("Part_17/data/featurized_pretrain.npz", pre_fps=pre_fps, pre_pIC50=pre_pIC50_arr, pre_valid_idx=np.array(pre_valid_idx), pre_tr_new=pre_tr_new, pre_val_new=pre_val_new)



## 5. Models — HybridRegression (pretrain) + HybridClassifier (finetune)


In [ ]:
class FPMLP(nn.Module):
    def __init__(self, in_dim=FP_DIM, hidden=[512,256,128], dropout=0.3):
        super().__init__()
        layers=[]
        d=in_dim
        for h in hidden:
            layers+=[nn.Linear(d,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d=h
        self.enc=nn.Sequential(*layers)
        self.out_dim=hidden[-1]
    def forward(self, fp): return self.enc(fp)

class GCNEncoder(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.dropout=dropout
        self.convs=nn.ModuleList([GCNConv(num_node_features if i==0 else hidden_dim, hidden_dim) for i in range(num_layers)])
        self.bns=nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers)])
        self.out_dim=hidden_dim*2
    def forward(self, x, edge_index, batch):
        for conv,bn in zip(self.convs, self.bns):
            x=F.relu(bn(conv(x, edge_index)))
            x=F.dropout(x, p=self.dropout, training=self.training)
        return torch.cat([global_mean_pool(x,batch), global_max_pool(x,batch)], dim=1)

class HybridRegression(nn.Module):
    def __init__(self, fp_dim=FP_DIM, gnn_hidden=128, gnn_layers=3, fp_hidden=[512,256,128], r=0.5, dropout=0.3):
        super().__init__()
        self.r=r
        self.fp_enc=FPMLP(fp_dim, fp_hidden, dropout)
        self.gnn_enc=GCNEncoder(78, gnn_hidden, gnn_layers, dropout)
        total=256
        g_dim=int(total*r); f_dim=total-g_dim
        self.g_proj=nn.Linear(self.gnn_enc.out_dim, g_dim)
        self.f_proj=nn.Linear(self.fp_enc.out_dim, f_dim)
        self.reg_head=nn.Sequential(nn.Linear(total,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,1))
    def forward(self, x, edge_index, batch, fp):
        g=self.gnn_enc(x, edge_index, batch); g=self.g_proj(g)
        f=self.fp_enc(fp); f=self.f_proj(f)
        h=torch.cat([g,f], dim=1)
        return self.reg_head(h).squeeze(-1)  # (B,)

class HybridClassifier(nn.Module):
    def __init__(self, fp_dim=FP_DIM, gnn_hidden=128, gnn_layers=3, fp_hidden=[512,256,128], r=0.5, dropout=0.3, num_classes=2):
        super().__init__()
        self.r=r
        self.fp_enc=FPMLP(fp_dim, fp_hidden, dropout)
        self.gnn_enc=GCNEncoder(78, gnn_hidden, gnn_layers, dropout)
        total=256
        g_dim=int(total*r); f_dim=total-g_dim
        self.g_proj=nn.Linear(self.gnn_enc.out_dim, g_dim)
        self.f_proj=nn.Linear(self.fp_enc.out_dim, f_dim)
        self.classifier=nn.Sequential(nn.Linear(total,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_classes))
    def forward(self, x, edge_index, batch, fp):
        g=self.gnn_enc(x, edge_index, batch); g=self.g_proj(g)
        f=self.fp_enc(fp); f=self.f_proj(f)
        h=torch.cat([g,f], dim=1)
        return self.classifier(h)
    def load_encoder_from_regression(self, reg_state):
        # copy encoder/proj weights from regression checkpoint
        own=self.state_dict()
        for k,v in reg_state.items():
            if k in own and 'reg_head' not in k:
                own[k].copy_(v)

class GNNOnly(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=3, dropout=0.3, num_classes=2):
        super().__init__()
        self.enc=GCNEncoder(78, hidden_dim, num_layers, dropout)
        self.cls=nn.Sequential(nn.Linear(self.enc.out_dim,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_classes))
    def forward(self,x,edge_index,batch): return self.cls(self.enc(x,edge_index,batch))

class FPOnly(nn.Module):
    def __init__(self, in_dim=FP_DIM, hidden=[512,256,128], dropout=0.3, num_classes=2):
        super().__init__()
        self.enc=FPMLP(in_dim, hidden, dropout)
        self.cls=nn.Sequential(nn.Linear(self.enc.out_dim,128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128,num_classes))
    def forward(self, fp): return self.cls(self.enc(fp))

print("models defined")



## 6. Train — pretrain regression on ChEMBL (pIC50)


In [ ]:
def train_epoch_reg(model, graphs, fps, pIC50s, idx, optimizer, batch_size=64):
    model.train()
    perm=np.random.permutation(idx)
    total_loss=0; n=0
    for s in range(0,len(perm),batch_size):
        bidx=perm[s:s+batch_size]
        b_graphs=[graphs[i] for i in bidx]
        b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
        b_y=torch.tensor(pIC50s[bidx], dtype=torch.float, device=device)
        loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
        batch=next(iter(loader)).to(device)
        optimizer.zero_grad()
        pred=model(batch.x, batch.edge_index, batch.batch, b_fp)
        loss=F.mse_loss(pred, b_y)
        loss.backward(); optimizer.step()
        total_loss+=loss.item()*len(bidx); n+=len(bidx)
    return total_loss/max(1,n)

def eval_reg(model, graphs, fps, pIC50s, idx):
    model.eval()
    preds=[]; trues=[]
    with torch.no_grad():
        for s in range(0,len(idx),128):
            bidx=idx[s:s+128]
            b_graphs=[graphs[i] for i in bidx]
            b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
            loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
            batch=next(iter(loader)).to(device)
            pred=model(batch.x, batch.edge_index, batch.batch, b_fp)
            preds.extend(pred.cpu().numpy().tolist())
            trues.extend(pIC50s[bidx].tolist())
    preds=np.array(preds); trues=np.array(trues)
    rmse=np.sqrt(mean_squared_error(trues, preds))
    r2=r2_score(trues, preds)
    mae=np.mean(np.abs(trues-preds))
    return preds, trues, {'rmse':rmse,'r2':r2,'mae':mae}

# load featurized pretrain
pack=np.load("Part_17/data/featurized_pretrain.npz")
pre_fps=pack['pre_fps']; pre_pIC50=pack['pre_pIC50']; pre_tr_new=pack['pre_tr_new']; pre_val_new=pack['pre_val_new']
# graphs already in memory from previous cell (pre_graphs) — but if kernel restart, rebuild is needed; we assume in-memory
print(f"pretrain: tr {len(pre_tr_new)} val {len(pre_val_new)} total graphs {len(pre_graphs)} fps {pre_fps.shape}")

# class for pretrain: same hybrid
pretrain_model=HybridRegression(r=0.5).to(device)
opt=torch.optim.Adam(pretrain_model.parameters(), lr=1e-3, weight_decay=1e-5)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
best_r2=-1; best_state=None
EPOCHS_PRE=80
for epoch in range(1, EPOCHS_PRE+1):
    loss=train_epoch_reg(pretrain_model, pre_graphs, pre_fps, pre_pIC50, pre_tr_new, opt, batch_size=64)
    if epoch%5==0 or epoch==1:
        _,_,m=eval_reg(pretrain_model, pre_graphs, pre_fps, pre_pIC50, pre_val_new)
        # also train metrics
        _,_,m_tr=eval_reg(pretrain_model, pre_graphs, pre_fps, pre_pIC50, pre_tr_new)
        sched.step(1-m['r2'] if not np.isnan(m['r2']) else 1)
        if m['r2']>best_r2:
            best_r2=m['r2']; best_state={k:v.cpu() for k,v in pretrain_model.state_dict().items()}
        print(f"ep{epoch:3d} loss {loss:.4f}  val r2 {m['r2']:.4f} rmse {m['rmse']:.4f} mae {m['mae']:.4f} | train r2 {m_tr['r2']:.4f} [best {best_r2:.4f}]")
if best_state is not None:
    pretrain_model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
torch.save(pretrain_model.state_dict(), "Part_17/models/hybrid_pretrain_regression.pth")
print(f"pretrain done best val r2 {best_r2:.4f} saved Part_17/models/hybrid_pretrain_regression.pth")
# final val
_,_,m_final=eval_reg(pretrain_model, pre_graphs, pre_fps, pre_pIC50, pre_val_new)
print("final val", m_final)



## 7. Finetune classification on curated 645 + controls (scratch hybrid, FP-only, GNN-only, RF)


In [ ]:
# helpers for classification
def train_epoch_hybrid_cls(model, graphs, fps, labels, idx, optimizer, criterion, batch_size=32):
    model.train()
    perm=np.random.permutation(idx)
    total=0; n=0
    for s in range(0,len(perm),batch_size):
        bidx=perm[s:s+batch_size]
        b_graphs=[graphs[i] for i in bidx]
        b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
        loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
        batch=next(iter(loader)).to(device)
        optimizer.zero_grad()
        out=model(batch.x, batch.edge_index, batch.batch, b_fp)
        loss=criterion(out, batch.y)
        loss.backward(); optimizer.step()
        total+=loss.item()*len(bidx); n+=len(bidx)
    return total/max(1,n)

def eval_hybrid_cls(model, graphs, fps, labels, idx):
    model.eval()
    sc=[]; tr=[]
    with torch.no_grad():
        for s in range(0,len(idx),64):
            bidx=idx[s:s+64]
            b_graphs=[graphs[i] for i in bidx]
            b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
            loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
            batch=next(iter(loader)).to(device)
            out=model(batch.x, batch.edge_index, batch.batch, b_fp)
            sc.extend(F.softmax(out, dim=1)[:,1].cpu().numpy().tolist())
            tr.extend(labels[bidx].tolist())
    return np.array(tr), np.array(sc)

def train_epoch_gnn(model, graphs, labels, idx, optimizer, criterion, batch_size=32):
    model.train()
    perm=np.random.permutation(idx)
    total=0; n=0
    for s in range(0,len(perm),batch_size):
        bidx=perm[s:s+batch_size]
        b_graphs=[graphs[i] for i in bidx]
        loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
        batch=next(iter(loader)).to(device)
        optimizer.zero_grad()
        out=model(batch.x, batch.edge_index, batch.batch)
        loss=criterion(out, batch.y)
        loss.backward(); optimizer.step()
        total+=loss.item()*len(bidx); n+=len(bidx)
    return total/max(1,n)
def eval_gnn(model, graphs, labels, idx):
    model.eval()
    sc=[]; tr=[]
    with torch.no_grad():
        for s in range(0,len(idx),64):
            bidx=idx[s:s+64]
            b_graphs=[graphs[i] for i in bidx]
            loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
            batch=next(iter(loader)).to(device)
            out=model(batch.x, batch.edge_index, batch.batch)
            sc.extend(F.softmax(out, dim=1)[:,1].cpu().numpy().tolist())
            tr.extend(labels[bidx].tolist())
    return np.array(tr), np.array(sc)

def train_epoch_fp(model, fps, labels, idx, optimizer, criterion, batch_size=64):
    model.train()
    perm=np.random.permutation(idx)
    total=0; n=0
    for s in range(0,len(perm),batch_size):
        bidx=perm[s:s+batch_size]
        b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
        y=torch.tensor(labels[bidx], dtype=torch.long, device=device)
        optimizer.zero_grad()
        out=model(b_fp)
        loss=criterion(out, y)
        loss.backward(); optimizer.step()
        total+=loss.item()*len(bidx); n+=len(bidx)
    return total/max(1,n)
def eval_fp(model, fps, labels, idx):
    model.eval()
    sc=[]; tr=[]
    with torch.no_grad():
        for s in range(0,len(idx),64):
            bidx=idx[s:s+64]
            b_fp=torch.tensor(fps[bidx], dtype=torch.float, device=device)
            out=model(b_fp)
            sc.extend(F.softmax(out, dim=1)[:,1].cpu().numpy().tolist())
            tr.extend(labels[bidx].tolist())
    return np.array(tr), np.array(sc)

# metrics helpers
def enrichment_factor(y_true, y_score, frac=0.01):
    n=len(y_true); k=max(1,int(n*frac))
    order=np.argsort(-y_score)
    hits=(y_true[order[:k]]==1).sum()
    base=(y_true==1).mean()
    return (hits/k)/base if base>0 else 0.0
def bedroc(y_true, y_score, alpha=20.0):
    n=len(y_true); n_act=(y_true==1).sum()
    if n_act==0 or n_act==n: return 0.0
    order=np.argsort(-y_score)
    y_sorted=y_true[order]
    ranks=np.where(y_sorted==1)[0]+1
    s=np.sum(np.exp(-alpha*ranks/n))
    s_max=np.sum(np.exp(-alpha*np.arange(1,n_act+1)/n))
    s_min=np.sum(np.exp(-alpha*np.arange(n-n_act+1, n+1)/n))
    if s_max==s_min: return 0.0
    return float(np.clip((s - s_min)/(s_max - s_min),0,1))
def all_metrics(y_true, y_score, thresh=0.5):
    from sklearn.metrics import precision_recall_curve, auc
    y_pred=(y_score>=thresh).astype(int)
    roc=roc_auc_score(y_true, y_score) if len(np.unique(y_true))>1 else 0.5
    prec, rec, _ = precision_recall_curve(y_true, y_score) if len(np.unique(y_true))>1 else ([0],[0],[0])
    pr_auc = auc(rec, prec) if len(prec)>1 else 0.0
    return {'roc_auc':roc,'pr_auc':pr_auc,'f1':f1_score(y_true,y_pred,zero_division=0),'acc':accuracy_score(y_true,y_pred),'mcc':matthews_corrcoef(y_true,y_pred) if len(np.unique(y_true))>1 else 0.0,'ef1':enrichment_factor(y_true,y_score,0.01),'ef5':enrichment_factor(y_true,y_score,0.05),'bedroc':bedroc(y_true,y_score,20.0)}



In [ ]:
# load curated featurized pack (if kernel restarted, reload)
import numpy as np, torch
pack=np.load("Part_17/data/featurized_curated.npz")
cur_fps=pack['cur_fps']; cur_labels=pack['cur_labels']; tr_cur_new=pack['tr_cur_new']; te_cur_new=pack['te_cur_new']; tr_rand_new=pack['tr_rand_new']; te_rand_new=pack['te_rand_new']
# also need graphs in memory; if not, rebuild quickly from df645 (fallback)
try:
    len(cur_graphs)
except NameError:
    print("rebuilding cur_graphs from df645...")
    # rebuild using same df645 loaded earlier
    cur_graphs=[]
    label_map={'Active':1,'Inactive':0}
    for i,row in df645.iloc[[int(i) for i in np.load("Part_17/data/featurized_curated.npz")['cur_valid_idx']]].iterrows():
        cur_graphs.append(mol_to_graph(row['canonical_smiles'], label=label_map[row['bioactivity_class']]))
    # filter None if any
    print("rebuilt", len(cur_graphs))

def run_finetune(name, tr_idx, te_idx, use_pretrain=True, r=0.5, epochs=100):
    print(f"\n===== finetune {name} use_pretrain={use_pretrain} r={r} tr {len(tr_idx)} te {len(te_idx)} =====")
    # class weights
    y_tr=cur_labels[tr_idx]
    n=len(y_tr); n1=(y_tr==1).sum(); n0=(y_tr==0).sum()
    w1=n/(2*max(1,n1)); w0=n/(2*max(1,n0))
    w=torch.tensor([w0,w1], dtype=torch.float, device=device)
    print(f" weights inactive {w0:.3f} active {w1:.3f} (active {(y_tr==1).mean()*100:.1f}% )")
    model=HybridClassifier(r=r).to(device)
    if use_pretrain:
        reg_state=torch.load("Part_17/models/hybrid_pretrain_regression.pth", map_location=device)
        model.load_encoder_from_regression(reg_state)
        print(" loaded pretrained encoder")
    crit=nn.CrossEntropyLoss(weight=w)
    opt=torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=10, factor=0.5)
    best_auc=0; best_state=None
    for epoch in range(1, epochs+1):
        loss=train_epoch_hybrid_cls(model, cur_graphs, cur_fps, cur_labels, tr_idx, opt, crit, batch_size=32)
        if epoch%10==0 or epoch==1:
            yt, ys = eval_hybrid_cls(model, cur_graphs, cur_fps, cur_labels, te_idx)
            m=all_metrics(yt, ys)
            sched.step(1-m['roc_auc'])
            if m['roc_auc']>best_auc:
                best_auc=m['roc_auc']; best_state={k:v.cpu() for k,v in model.state_dict().items()}
            print(f" ep{epoch:3d} loss {loss:.4f} roc {m['roc_auc']:.4f} pr {m['pr_auc']:.4f} ef1 {m['ef1']:.2f} bedroc {m['bedroc']:.3f} f1 {m['f1']:.3f} [best {best_auc:.4f}]")
    if best_state is not None:
        model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
    tag=f"{name}_{'pretrain' if use_pretrain else 'scratch'}_r{r}"
    torch.save(model.state_dict(), f"Part_17/models/hybrid_{tag}.pth")
    yt, ys = eval_hybrid_cls(model, cur_graphs, cur_fps, cur_labels, te_idx)
    m_final=all_metrics(yt, ys)
    print(f"final {tag}:", m_final)
    return m_final, (yt, ys)

# A) scaffold — main result
metrics_scaffold_pretrain, scores_scaffold_pretrain = run_finetune("scaffold", tr_cur_new, te_cur_new, use_pretrain=True, r=0.5, epochs=120)
metrics_scaffold_scratch, scores_scaffold_scratch = run_finetune("scaffold", tr_cur_new, te_cur_new, use_pretrain=False, r=0.5, epochs=120)
# B) random for comparability
metrics_random_pretrain, scores_random_pretrain = run_finetune("random", tr_rand_new, te_rand_new, use_pretrain=True, r=0.5, epochs=120)
metrics_random_scratch, _ = run_finetune("random", tr_rand_new, te_rand_new, use_pretrain=False, r=0.5, epochs=120)

# Also train FP-only / GNN-only / RF on scaffold for context
print("\n--- controls on scaffold ---")
# FP-only
fp_model=FPOnly().to(device)
w_fp=torch.tensor([ len(cur_labels[tr_cur_new])/(2*max(1,(cur_labels[tr_cur_new]==0).sum())), len(cur_labels[tr_cur_new])/(2*max(1,(cur_labels[tr_cur_new]==1).sum())) ], dtype=torch.float, device=device)
opt_fp=torch.optim.Adam(fp_model.parameters(), lr=1e-3)
sched_fp=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_fp, patience=10, factor=0.5)
crit_fp=nn.CrossEntropyLoss(weight=w_fp)
best=0; bs=None
for epoch in range(1,121):
    train_epoch_fp(fp_model, cur_fps, cur_labels, tr_cur_new, opt_fp, crit_fp)
    if epoch%10==0:
        yt, ys = eval_fp(fp_model, cur_fps, cur_labels, te_cur_new)
        m=all_metrics(yt, ys); sched_fp.step(1-m['roc_auc'])
        if m['roc_auc']>best: best=m['roc_auc']; bs={k:v.cpu() for k,v in fp_model.state_dict().items()}
if bs is not None: fp_model.load_state_dict({k:v.to(device) for k,v in bs.items()})
torch.save(fp_model.state_dict(), "Part_17/models/fp_only_scaffold.pth")
yt_fp, ys_fp = eval_fp(fp_model, cur_fps, cur_labels, te_cur_new)
m_fp=all_metrics(yt_fp, ys_fp)
print("FP-only scaffold", m_fp)
# GNN-only
gnn_model=GNNOnly().to(device)
opt_g=torch.optim.Adam(gnn_model.parameters(), lr=1e-3)
sched_g=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_g, patience=10, factor=0.5)
crit_g=nn.CrossEntropyLoss(weight=w_fp)
best=0; bs=None
for epoch in range(1,121):
    train_epoch_gnn(gnn_model, cur_graphs, cur_labels, tr_cur_new, opt_g, crit_g)
    if epoch%10==0:
        yt, ys = eval_gnn(gnn_model, cur_graphs, cur_labels, te_cur_new)
        m=all_metrics(yt, ys); sched_g.step(1-m['roc_auc'])
        if m['roc_auc']>best: best=m['roc_auc']; bs={k:v.cpu() for k,v in gnn_model.state_dict().items()}
if bs is not None: gnn_model.load_state_dict({k:v.to(device) for k,v in bs.items()})
torch.save(gnn_model.state_dict(), "Part_17/models/gnn_only_scaffold.pth")
yt_g, ys_g = eval_gnn(gnn_model, cur_graphs, cur_labels, te_cur_new)
m_g=all_metrics(yt_g, ys_g)
print("GNN-only scaffold", m_g)
# RF
rf=RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight='balanced')
rf.fit(cur_fps[tr_cur_new], cur_labels[tr_cur_new])
ys_rf=rf.predict_proba(cur_fps[te_cur_new])[:,1]
m_rf=all_metrics(cur_labels[te_cur_new], ys_rf)
print("RF scaffold", m_rf)
import pickle
with open("Part_17/models/rf_scaffold.pkl","wb") as f: pickle.dump(rf,f)

# aggregate
import pandas as pd
rows=[]
for split, d in [("scaffold_pretrain",metrics_scaffold_pretrain),("scaffold_scratch",metrics_scaffold_scratch),("random_pretrain",metrics_random_pretrain),("random_scratch",metrics_random_scratch),("scaffold_fp",m_fp),("scaffold_gnn",m_g),("scaffold_rf",m_rf)]:
    row={"model":split}
    row.update(d)
    rows.append(row)
df_metrics=pd.DataFrame(rows)
df_metrics.to_csv("Part_17/results/metrics.csv", index=False)
print(df_metrics.to_string(index=False))
print("saved Part_17/results/metrics.csv")

# bar plot
import matplotlib.pyplot as plt
fig, ax=plt.subplots(figsize=(10,4))
x=np.arange(len(df_metrics))
ax.bar(x-0.2, df_metrics['roc_auc'], 0.2, label='ROC-AUC', color='steelblue')
ax.bar(x, df_metrics['bedroc'], 0.2, label='BEDROC', color='coral')
ax.bar(x+0.2, df_metrics['ef1'], 0.2, label='EF@1%', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(df_metrics['model'], rotation=20, ha='right')
ax.legend()
plt.tight_layout(); plt.savefig("Part_17/results/ablation.png", dpi=200); plt.show()
print("saved Part_17/results/ablation.png")



## 8. Calibration (reliability)


In [ ]:
def reliability(y_true, y_score, bins=10):
    edges=np.linspace(0,1,bins+1)
    xs=[]; ys=[]; ns=[]
    for i in range(bins):
        m=(y_score>=edges[i])&(y_score<edges[i+1])
        if i==bins-1: m=(y_score>=edges[i])&(y_score<=edges[i+1])
        if m.sum()==0: xs.append((edges[i]+edges[i+1])/2); ys.append(0); ns.append(0)
        else: xs.append(y_score[m].mean()); ys.append(y_true[m].mean()); ns.append(m.sum())
    return np.array(xs), np.array(ys), np.array(ns)

import matplotlib.pyplot as plt
fig, axes=plt.subplots(1,2, figsize=(11,4))
for ax, (name, te_idx, tag) in zip(axes, [("scaffold", te_cur_new, "scaffold_pretrain_r0.5"), ("scaffold-scratch", te_cur_new, "scaffold_scratch_r0.5")]):
    # reload best
    hyp=HybridClassifier(r=0.5).to(device)
    p=f"Part_17/models/hybrid_{tag}.pth"
    if os.path.exists(p):
        hyp.load_state_dict(torch.load(p, map_location=device))
        hyp.eval()
        yt, ys = eval_hybrid_cls(hyp, cur_graphs, cur_fps, cur_labels, te_idx)
        xs, ys_rel, ns = reliability(yt, ys, 10)
        ax.plot([0,1],[0,1],"k--", label="perfect")
        ax.plot(xs, ys_rel, marker="o", label=tag)
        ax.set_title(f"calibration {name}"); ax.set_xlabel("mean p"); ax.set_ylabel("fraction positive"); ax.legend()
plt.tight_layout(); plt.savefig("Part_17/results/calibration.png", dpi=200); plt.show()
print("saved Part_17/results/calibration.png")



## 9. COCONUT screening — domain filter on curated stats, top-k (not just threshold)


In [ ]:
# locate COCONUT
cands=["Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/screening_results.csv","Part_8/screening_results.csv","../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/screening_results.csv"]
coconut_path=None
for c in cands:
    if os.path.exists(c):
        coconut_path=c; break
if not coconut_path:
    import glob
    for f in glob.glob("**/screening_results.csv", recursive=True):
        coconut_path=f; break
print(f"COCONUT {coconut_path}")
coconut=pd.read_csv(coconut_path)
print(len(coconut), list(coconut.columns)[:5])
smiles_col='canonical_smiles' if 'canonical_smiles' in coconut.columns else coconut.columns[1]
print(f"SMILES col {smiles_col}")

# domain: curated 645 stats (finetune domain, not pretrain broad)
# recompute MW/logp if not present
from rdkit.Chem import Descriptors, Lipinski
if 'mw' not in coconut.columns:
    mws=[]; logps=[]; hbas=[]; hbds=[]
    for smi in tqdm(coconut[smiles_col], desc='COCONUT props'):
        m=Chem.MolFromSmiles(str(smi))
        if m is None: mws.append(np.nan); logps.append(np.nan); hbas.append(np.nan); hbds.append(np.nan)
        else: mws.append(Descriptors.MolWt(m)); logps.append(Descriptors.MolLogP(m)); hbas.append(Lipinski.NumHAcceptors(m)); hbds.append(Lipinski.NumHDonors(m))
    coconut['mw']=mws; coconut['logp']=logps; coconut['n_hba']=hbas; coconut['n_hbd']=hbds

# curated stats
t_mw=df645['molecular_weight'].values; t_logp=df645['logp'].values; t_hba=df645['n_hba'].values; t_hbd=df645['n_hbd'].values
def bounds(a): return (np.nanmean(a)-2*np.nanstd(a), np.nanmean(a)+2*np.nanstd(a))
mw_lo,mw_hi=bounds(t_mw); logp_lo,logp_hi=bounds(t_logp); hba_lo,hba_hi=bounds(t_hba); hbd_lo,hbd_hi=bounds(t_hbd)
print(f"domain MW [{mw_lo:.1f},{mw_hi:.1f}] LogP [{logp_lo:.2f},{logp_hi:.2f}] HBA [{hba_lo:.1f},{hba_hi:.1f}] HBD [{hbd_lo:.1f},{hbd_hi:.1f}]")
coconut['in_domain']=(coconut['mw']>=mw_lo)&(coconut['mw']<=mw_hi)&(coconut['logp']>=logp_lo)&(coconut['logp']<=logp_hi)&(coconut['n_hba']>=hba_lo)&(coconut['n_hba']<=hba_hi)&(coconut['n_hbd']>=hbd_lo)&(coconut['n_hbd']<=hbd_hi)&coconut['mw'].notna()
print(f"in-domain {coconut['in_domain'].sum()} ({coconut['in_domain'].mean()*100:.1f}%)")
coconut.to_csv("Part_17/results/coconut_with_domain.csv", index=False)

# featurize in-domain
in_idx=coconut[coconut['in_domain']].index.tolist()
print(f"featurizing {len(in_idx)}")
cocos_fp=[]; cocos_graphs=[]; cocos_valid_idx=[]
for idx in tqdm(in_idx, desc='COCONUT featurize'):
    smi=coconut.at[idx, smiles_col]
    fp=combined_fp(smi)
    g=mol_to_graph(smi, label=0)
    if fp is not None and g is not None:
        cocos_fp.append(fp); cocos_graphs.append(g); cocos_valid_idx.append(idx)
cocos_fp=np.stack(cocos_fp) if cocos_fp else np.zeros((0,FP_DIM), dtype=np.float32)
print(f"valid {len(cocos_valid_idx)}")



In [ ]:
# screen with best scaffold-pretrained model (and also scratch for comparison)
import torch.nn.functional as F
def screen_with(tag):
    p=f"Part_17/models/hybrid_{tag}.pth"
    print(f"screen {p} exists={os.path.exists(p)}")
    m=HybridClassifier(r=0.5).to(device)
    m.load_state_dict(torch.load(p, map_location=device))
    m.eval()
    probs=[]
    with torch.no_grad():
        for s in tqdm(range(0,len(cocos_valid_idx),256), desc=f'screen {tag}'):
            b_fp=torch.tensor(cocos_fp[s:s+256], dtype=torch.float, device=device)
            b_graphs=cocos_graphs[s:s+256]
            from torch_geometric.loader import DataLoader
            loader=DataLoader(b_graphs, batch_size=len(b_graphs), shuffle=False)
            batch=next(iter(loader)).to(device)
            out=m(batch.x, batch.edge_index, batch.batch, b_fp)
            probs.extend(F.softmax(out, dim=1)[:,1].cpu().numpy().tolist())
    return np.array(probs)

probs_pretrain=screen_with("scaffold_pretrain_r0.5")
probs_scratch=screen_with("scaffold_scratch_r0.5")

# build results
res=coconut.loc[cocos_valid_idx, ['mw','logp','n_hba','n_hbd']].copy()
res['canonical_smiles']=coconut.loc[cocos_valid_idx, smiles_col].values
res['identifier']=coconut.loc[cocos_valid_idx, 'identifier'].values if 'identifier' in coconut.columns else cocos_valid_idx
res['hybrid_pretrain_prob']=probs_pretrain
res['hybrid_scratch_prob']=probs_scratch
# also RF for comparison
import pickle
rf_path="Part_17/models/rf_scaffold.pkl"
if os.path.exists(rf_path):
    with open(rf_path,"rb") as f: rf=pickle.load(f)
    res['rf_prob']=rf.predict_proba(cocos_fp)[:,1]
res.to_csv("Part_17/results/coconut_hybrid_screening.csv", index=False)
print(f"saved {len(res)}")
print(f"pretrain mean {probs_pretrain.mean():.4f} max {probs_pretrain.max():.4f}")
print(f"scratch  mean {probs_scratch.mean():.4f} max {probs_scratch.max():.4f}")

print("\n=== threshold sweep pretrain ===")
for t in [0.5,0.6,0.7,0.8,0.85,0.9]:
    n=(probs_pretrain>t).sum()
    print(f" p>{t:.2f}: {n:5d} ({n/len(probs_pretrain)*100:5.2f}%)")
print("\n=== top-k pretrain ===")
order=np.argsort(-probs_pretrain)
for k in [10,50,100,500,1000]:
    print(f" top {k}: min prob {probs_pretrain[order[k-1]]:.4f}")

# save hits by threshold and top-k
for t in [0.85,0.9]:
    sel=res[res['hybrid_pretrain_prob']>t].sort_values('hybrid_pretrain_prob', ascending=False)
    sel.to_csv(f"Part_17/results/coconut_hits_pretrain_{t}.csv", index=False)
    print(f"hits p>{t}: {len(sel)}")

for k in [100,500,1000]:
    sel=res.iloc[order[:k]].sort_values('hybrid_pretrain_prob', ascending=False)
    sel.to_csv(f"Part_17/results/coconut_top{k}_pretrain.csv", index=False)
    print(f"top{k} saved")

# also scratch hits
for t in [0.85]:
    sel=res[res['hybrid_scratch_prob']>t].sort_values('hybrid_scratch_prob', ascending=False)
    sel.to_csv(f"Part_17/results/coconut_hits_scratch_{t}.csv", index=False)
    print(f"scratch hits p>{t}: {len(sel)}")



## 10. Hit analysis — RF-116 overlap + Tanimoto novelty


In [ ]:
rf_hits_path=None
for cand in ["Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv","Part_8/filtered_compounds.csv","../Natural_MDM2_Inhibitor_Discovery_using_ML/Part_8/filtered_compounds.csv"]:
    if os.path.exists(cand):
        rf_hits_path=cand; break
if rf_hits_path and os.path.exists(rf_hits_path):
    rf_hits=pd.read_csv(rf_hits_path)
    print(f"RF 116 {len(rf_hits)}")
    rf_smi=set(rf_hits['canonical_smiles'].astype(str)) if 'canonical_smiles' in rf_hits.columns else set(rf_hits.iloc[:,0].astype(str))
    for t in [0.85]:
        p=f"Part_17/results/coconut_hits_pretrain_{t}.csv"
        if os.path.exists(p):
            h=pd.read_csv(p)
            inter=len(rf_smi & set(h['canonical_smiles'].astype(str)))
            print(f"pretrain p>{t} {len(h)} overlap RF116 {inter} novel {len(h)-inter}")
    for k in [100,500]:
        p=f"Part_17/results/coconut_top{k}_pretrain.csv"
        if os.path.exists(p):
            h=pd.read_csv(p)
            inter=len(rf_smi & set(h['canonical_smiles'].astype(str)))
            print(f"top{k} pretrain overlap {inter} novel {len(h)-inter}")

# Tanimoto novelty: max similarity to curated actives
train_actives=df645[df645['bioactivity_class']=='Active']['canonical_smiles'].tolist()
from rdkit.Chem import rdFingerprintGenerator as _rfg
_tani_gen = _rfg.GetMorganGenerator(radius=2, fpSize=2048)
train_fps=[_tani_gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in train_actives if Chem.MolFromSmiles(s) is not None]
print(f"train active fps {len(train_fps)}")
for fname in ["Part_17/results/coconut_top100_pretrain.csv","Part_17/results/coconut_top500_pretrain.csv"]:
    if not os.path.exists(fname): continue
    h=pd.read_csv(fname)
    if len(h)==0: continue
    max_sims=[]
    for smi in tqdm(h['canonical_smiles'], desc=f"Tanimoto {fname}"):
        m=Chem.MolFromSmiles(str(smi))
        if m is None: max_sims.append(0)
        else:
            fp=_tani_gen.GetFingerprint(m)
            sims=DataStructs.BulkTanimotoSimilarity(fp, train_fps)
            max_sims.append(max(sims) if sims else 0)
    h['max_tanimoto_to_curated_active']=max_sims
    h.to_csv(fname, index=False)
    print(f"{fname} mean max Tanimoto {np.mean(max_sims):.3f} <0.4 {(np.array(max_sims)<0.4).sum()}/{len(max_sims)}")
    print(h[['identifier','hybrid_pretrain_prob','max_tanimoto_to_curated_active']].head(5).to_string(index=False))



## 11. Screening plots


In [ ]:
import matplotlib.pyplot as plt
fig, axes=plt.subplots(1,3, figsize=(15,4))
axes[0].hist(probs_pretrain, bins=50, color='steelblue', edgecolor='black', alpha=0.7, label='pretrain')
axes[0].hist(probs_scratch, bins=50, color='coral', edgecolor='black', alpha=0.5, label='scratch')
axes[0].axvline(0.85, color='red', linestyle='--', label='0.85')
axes[0].set_title('p(active) in-domain COCONUT'); axes[0].set_xlabel('p'); axes[0].legend()
if 'rf_prob' in res.columns:
    axes[1].scatter(res['rf_prob'], res['hybrid_pretrain_prob'], s=2, alpha=0.3)
    axes[1].plot([0,1],[0,1],'r--')
    axes[1].set_xlabel('RF prob'); axes[1].set_ylabel('Pretrain Hybrid prob'); axes[1].set_title('RF vs Pretrain')
else:
    axes[1].hist(res['mw'], bins=40, color='coral', edgecolor='black')
    axes[1].set_title('MW screened'); axes[1].set_xlabel('Da')
axes[2].scatter(res['mw'], res['logp'], c=res['hybrid_pretrain_prob'], cmap='RdYlGn', s=3, alpha=0.6)
axes[2].set_xlabel('MW'); axes[2].set_ylabel('LogP'); axes[2].set_title('COCONUT colored by pretrain p')
plt.tight_layout(); plt.savefig('Part_17/results/screening_analysis.png', dpi=200); plt.show()
print("saved Part_17/results/screening_analysis.png")



## 12. Summary


In [ ]:
print("="*60)
print("PART 17 COMPLETE")
print("="*60)
# curated stats
print(f"Curated 645: train {len(tr_cur_new)} test {len(te_cur_new)} (scaffold)")
print(f"Pretrain 4441 after exclusion: {len(pre_valid_idx)} valid → tr {len(pre_tr_new)} val {len(pre_val_new)}")
print(f"Finetune scaffold pretrain ROC {metrics_scaffold_pretrain['roc_auc']:.4f} vs scratch {metrics_scaffold_scratch['roc_auc']:.4f} vs RF {m_rf['roc_auc']:.4f}")
print(f"Pretrain val r2 {m_final['r2']:.4f} rmse {m_final['rmse']:.4f}")
print(f"Screen in-domain {len(cocos_valid_idx)} pretrain mean {probs_pretrain.mean():.4f} max {probs_pretrain.max():.4f} hits>0.85 {(probs_pretrain>0.85).sum()} top100 min {np.sort(probs_pretrain)[-100]:.4f}")
print("\nOutputs:")
for p in ["Part_17/data/splits_curated.npz","Part_17/data/splits_pretrain.npz","Part_17/models/hybrid_pretrain_regression.pth","Part_17/models/hybrid_scaffold_pretrain_r0.5.pth","Part_17/results/metrics.csv","Part_17/results/coconut_hybrid_screening.csv","Part_17/results/coconut_top100_pretrain.csv","Part_17/results/screening_analysis.png"]:
    print(f"  {p} {'OK' if os.path.exists(p) else 'MISSING'}")
print("\nDecision: if scaffold_pretrain BEDROC/EF@1% >= RF and top hits novel + sane rate, pretrain helps generalization.")

